# 1. 인공지능기본법 구조 기반 Document Chunking

- 참고 실습: `track1_core/03-1_document_chunking.ipynb`
- 적용 데이터: `data/samples/인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx`
- 핵심 전략: HWPX 파싱, 본칙/부칙 분리, 장·조·항·호·목 계층 보존, Parent-Child Chunking


## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. 문서 전처리와 Chunking이 왜 필요한지 설명할 수 있다.
2. 고정 길이, Overlap, 구조 기반 Chunking의 차이를 비교할 수 있다.
3. Chunk Size와 Overlap이 문맥 손실, 주제 혼합, 검색 정밀도에 미치는 영향을 설명할 수 있다.
4. 실제 RAG에서는 `RecursiveCharacterTextSplitter`를 사용해 Chunk를 생성할 수 있다.
5. 선택·심화 학습으로 문장/문단 직접 구현과 Parent-Child 구조의 용도를 설명할 수 있다.


## 2. 문제 상황

법률 문서를 일반 보고서처럼 일정 글자 수로 자르면 `제34조`라는 근거 조문과
그 조문의 항·호가 분리되거나, 본칙 `제1조`와 부칙 `제1조`가 같은 ID로 충돌할 수
있다. 이 실습은 HWPX에서 법령 본문을 추출한 뒤 본칙/부칙, 장, 조, 항, 호, 목의
계층을 인식한다. 짧은 조문은 전체를 유지하고 긴 조문만 하위 법률 단위 경계에서
나눠 검색 정밀도와 법적 문맥을 함께 보존한다.


## 3. 핵심 개념

법률 RAG의 기본 검색 단위는 **조문**이다. 조문 번호와 제목은 인용의 기준이고,
항·호·목은 조문 안에서 의무·요건·예외를 구체화한다.

| 계층 | 예시 | Chunking 원칙 |
|---|---|---|
| 법률 | 인공지능기본법 | 모든 Chunk의 공통 출처 metadata |
| 본칙/부칙 | 본칙, 부칙 | ID 공간을 분리해 같은 조 번호 충돌 방지 |
| 장 | 제4장 인공지능윤리 및 신뢰성 확보 | 검색 Filter와 상위 문맥으로 보존 |
| 조 | 제34조(고영향 인공지능과 관련한 사업자의 책무) | 기본 Parent 및 검색 단위 |
| 항 | ①, ② | 긴 조문의 1차 분할 경계 |
| 호 | 1., 2. | 긴 항의 2차 분할 경계 |
| 목 | 가., 나. | 긴 호의 3차 분할 경계 |

`max_chars=1000`은 강제 절단 위치가 아니라 **장문 조문을 세분화할지 결정하는
임계값**이다. 실제 절단은 항→호→목 경계에서만 수행한다. 각 Child에는 법률명,
본칙/부칙, 장, 조문 표제와 `parent_id`를 반복해 단독 검색되어도 출처를 잃지 않게 한다.


## 4. 실행 구조

```text
HWPX Contents/section*.xml
  → 문단 추출 및 반복 머리글 제외(첫 장부터 시작)
  → 본칙/부칙·장·조문 파싱
  → 조문 Parent 47개 생성
  → 1,000자 이하: 조문 전체를 검색 Chunk로 유지
  → 1,000자 초과: 항 → 호 → 목 경계로만 세분화
  → 법률 계층 metadata 및 parent_id 검증
```


## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [3]:
import re
import zipfile
from xml.etree import ElementTree as ET

import pandas as pd

from agentic_ai.logging_utils import save_log
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import OUTPUT_DIR, PROJECT_ROOT, data_path

print_environment_summary()

DATA_PATH = data_path(
    "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx",
    must_exist=True,
)
LAW_ID = "ai-trust-basic-act-20676"
LAW_NAME = "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법"
SOURCE_FILE = DATA_PATH.name
COLLECTION_NAME = "aitrust_law_chunks_v1"

HP_NS = "http://www.hancom.co.kr/hwpml/2011/paragraph"
XML_NS = {"hp": HP_NS}


def load_hwpx_paragraphs(file_path) -> list[str]:
    """HWPX의 section XML에서 화면에 표시되는 문단 텍스트를 순서대로 추출한다."""
    paragraphs: list[str] = []
    with zipfile.ZipFile(file_path) as archive:
        section_names = sorted(
            name
            for name in archive.namelist()
            if re.fullmatch(r"Contents/section\d+\.xml", name)
        )
        if not section_names:
            raise ValueError(f"HWPX 본문 section XML을 찾을 수 없습니다: {file_path}")

        for section_name in section_names:
            root = ET.fromstring(archive.read(section_name))
            for paragraph in root.findall(".//hp:p", XML_NS):
                raw_text = "".join(
                    node.text or "" for node in paragraph.findall(".//hp:t", XML_NS)
                )
                normalized = re.sub(r"\s+", " ", raw_text).strip()
                if normalized:
                    paragraphs.append(normalized)
    return paragraphs


RAW_PARAGRAPHS = load_hwpx_paragraphs(DATA_PATH)
RAW_TEXT = "\n".join(RAW_PARAGRAPHS)
effective_date_match = re.search(r"\[시행\s+([^\]]+)\]", RAW_TEXT)
LAW_EFFECTIVE_DATE = effective_date_match.group(1) if effective_date_match else "확인 필요"

first_chapter_index = next(
    index for index, paragraph in enumerate(RAW_PARAGRAPHS)
    if re.match(r"^제\d+장\s+", paragraph)
)
LAW_BODY_PARAGRAPHS = RAW_PARAGRAPHS[first_chapter_index:]
document_text = "\n".join(LAW_BODY_PARAGRAPHS)

print(f"입력 파일: {DATA_PATH}")
print(f"전체 HWPX 문단: {len(RAW_PARAGRAPHS)}개")
print(f"법령 본문 문단: {len(LAW_BODY_PARAGRAPHS)}개 / {len(document_text):,}자")
print(f"시행일: {LAW_EFFECTIVE_DATE}")


[환경 설정 확인]
- 프로젝트: E:\agentic_ai_lab
- 데이터: E:\agentic_ai_lab\data
- 출력: E:\agentic_ai_lab\outputs
입력 파일: E:\agentic_ai_lab\data\인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx
전체 HWPX 문단: 390개
법령 본문 문단: 379개 / 24,798자
시행일: 2026. 1. 22.


## 6. 최소 실행 예제

전체 구현에 앞서, Chunking의 가장 단순한 형태를 확인한다. 마침표(`.`)로만 나누는
초간단 버전이다.


In [4]:
mini_law = [
    "제34조(고영향 인공지능과 관련한 사업자의 책무) ① 사업자는 필요한 조치를 이행하여야 한다.",
    "1. 위험관리방안의 수립ㆍ운영",
    "2. 이용자 보호 방안의 수립ㆍ운영",
]
print("\n".join(mini_law))
print("→ 조문 표제와 항·호를 함께 보존해야 각 의무의 근거를 알 수 있다.")


제34조(고영향 인공지능과 관련한 사업자의 책무) ① 사업자는 필요한 조치를 이행하여야 한다.
1. 위험관리방안의 수립ㆍ운영
2. 이용자 보호 방안의 수립ㆍ운영
→ 조문 표제와 항·호를 함께 보존해야 각 의무의 근거를 알 수 있다.


## 7. 단계별 구현

### 7.1 공통 Chunk 데이터 구조

일반 비교 전략은 `make_chunk()`를 사용한다. 법률 구조 기반 전략은 여기에
`scope`, `chapter`, `article`, `article_title`, `chunk_type`, `hierarchy_path`,
`parent_id`를 추가한다.


In [5]:
def make_chunk(chunk_id: str, document_id: str, section: str | None, text: str, parent_id: str | None = None) -> dict:
    """모든 Chunking 전략이 공유하는 표준 Chunk 형식을 만든다."""
    return {
        "chunk_id": chunk_id,
        "document_id": document_id,
        "section": section,
        "text": text,
        "length": len(text),
        "parent_id": parent_id,
    }


### 7.2 문장 단위 Chunking [선택: 원리 확인]

문장 부호(`.`, `!`, `?`) 뒤의 공백을 기준으로 나눈다.


In [6]:
# 마침표 자체는 보존하고, 문장 부호 바로 뒤의 공백만 분리 경계로 사용한다.
_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")


def chunk_by_sentence(text: str, document_id: str) -> list[dict]:
    """문장 부호를 기준으로 문장 단위 Chunk를 만든다."""
    sentences = [s.strip() for s in _SENTENCE_END.split(text.strip()) if s.strip()]
    return [
        make_chunk(f"{document_id}-sent-{i}", document_id, None, s)
        for i, s in enumerate(sentences, start=1)
    ]


sentence_chunks = chunk_by_sentence(document_text, LAW_ID)
print(f"문장 단위 Chunk 수: {len(sentence_chunks)}")
print(sentence_chunks[0])


문장 단위 Chunk 수: 417
{'chunk_id': 'ai-trust-basic-act-20676-sent-1', 'document_id': 'ai-trust-basic-act-20676', 'section': None, 'text': '제1장 총칙\n제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.', 'length': 123, 'parent_id': None}


**관찰**: 문장 부호만으로 나누면 조문 표제와 후속 항·호가 서로 다른 Chunk로
흩어진다. 법률 인용과 요건 판단에는 적합하지 않으므로 비교 기준으로만 사용한다.


### 7.3 고정 길이 Chunking [필수]

글자 수(`chunk_size`)를 기준으로 문서를 순서대로 자른다.


In [7]:
def chunk_by_fixed_length(text: str, document_id: str, chunk_size: int) -> list[dict]:
    """chunk_size 글자 단위로 문서를 순서대로 자른다."""
    if chunk_size <= 0:
        raise ValueError("chunk_size는 1 이상이어야 합니다.")
    text = text.strip()
    chunks = []
    # range의 start 값을 slice 시작점으로 사용하므로 겹침 없이 일정 간격으로 이동한다.
    for i, start in enumerate(range(0, len(text), chunk_size), start=1):
        piece = text[start:start + chunk_size]
        chunks.append(make_chunk(f"{document_id}-fixed-{i}", document_id, None, piece))
    return chunks


fixed_chunks = chunk_by_fixed_length(document_text, "ai-trust-basic-act-20676", chunk_size=150)
print(f"고정 길이(150자) Chunk 수: {len(fixed_chunks)}")
print(fixed_chunks[0])


고정 길이(150자) Chunk 수: 166
{'chunk_id': 'ai-trust-basic-act-20676-fixed-1', 'document_id': 'ai-trust-basic-act-20676', 'section': None, 'text': '제1장 총칙\n제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.\n제2조(정의) 이 법에서 사용하는 용어의 뜻은 ', 'length': 150, 'parent_id': None}


### 7.4 Overlap Chunking [필수]

**TODO**: `chunk_by_fixed_length_with_overlap()`을 작성한다.

- `overlap`은 `chunk_size`보다 작아야 한다. 그렇지 않으면 `ValueError`를 발생시킨다.
- 이동 거리(`step`)는 `chunk_size - overlap`이다.
- `start`를 0부터 `step`씩 늘려가며 `text[start:start + chunk_size]`를 자른다.
- `start`가 `len(text)` 이상이 되면 반복을 멈춘다.
- 각 Chunk는 `make_chunk(f"{document_id}-overlap-{i}", document_id, None, piece)`로
  만든다. `i`는 1부터 시작한다.

예상 출력: `chunk_by_fixed_length_with_overlap("가나다라마바사", "d", chunk_size=4, overlap=1)`의
첫 번째 Chunk `text`는 `"가나다라"`, 두 번째 Chunk `text`는 `"라마바사"`이다(3글자씩
이동, 마지막 글자 `"라"`가 겹침).


In [8]:
def chunk_by_fixed_length_with_overlap(text: str, document_id: str, chunk_size: int, overlap: int) -> list[dict]:
    """chunk_size 글자 단위로 자르되, overlap 글자만큼 앞 Chunk와 겹치게 만든다."""
    if chunk_size <= 0:
        raise ValueError("chunk_size는 1 이상이어야 합니다.")
    if overlap < 0:
        raise ValueError("overlap은 0 이상이어야 합니다.")
    if overlap >= chunk_size:
        raise ValueError("overlap은 chunk_size보다 작아야 합니다.")
    text = text.strip()
    # 다음 시작점을 chunk_size가 아니라 차이만큼 옮겨 overlap 구간을 다시 포함한다.
    step = chunk_size - overlap
    chunks = []
    i = 1
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size]
        chunks.append(make_chunk(f"{document_id}-overlap-{i}", document_id, None, piece))
        # 마지막 Chunk가 문서 끝에 닿으면 중복된 짧은 꼬리 Chunk를 만들지 않고 종료한다.
        if start + chunk_size >= len(text):
            break
        start += step
        i += 1
    return chunks


demo_overlap = chunk_by_fixed_length_with_overlap("가나다라마바사", "d", chunk_size=4, overlap=1)
print([c["text"] for c in demo_overlap])

overlap_chunks = chunk_by_fixed_length_with_overlap(document_text, "ai-trust-basic-act-20676", chunk_size=150, overlap=30)
print(f"Overlap(150자, overlap 30) Chunk 수: {len(overlap_chunks)}")


['가나다라', '라마바사']
Overlap(150자, overlap 30) Chunk 수: 207


### 7.5 HWPX 문단 단위 Chunking [비교]

HWPX의 화면 문단은 항·호·목과 대체로 대응하지만, 문단만 단독 저장하면 어느 장과
조문에 속하는지 알 수 없다. 따라서 문단 경계는 법률 계층 파서 안에서 사용한다.


In [9]:
def chunk_by_paragraph(text: str, document_id: str) -> list[dict]:
    paragraphs = [paragraph.strip() for paragraph in text.splitlines() if paragraph.strip()]
    return [
        make_chunk(f"{document_id}-para-{index}", document_id, None, paragraph)
        for index, paragraph in enumerate(paragraphs, start=1)
    ]


paragraph_chunks = chunk_by_paragraph(document_text, LAW_ID)
print(f"HWPX 본문 문단 Chunk 수: {len(paragraph_chunks)}")
print(paragraph_chunks[:3])


HWPX 본문 문단 Chunk 수: 379
[{'chunk_id': 'ai-trust-basic-act-20676-para-1', 'document_id': 'ai-trust-basic-act-20676', 'section': None, 'text': '제1장 총칙', 'length': 6, 'parent_id': None}, {'chunk_id': 'ai-trust-basic-act-20676-para-2', 'document_id': 'ai-trust-basic-act-20676', 'section': None, 'text': '제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.', 'length': 116, 'parent_id': None}, {'chunk_id': 'ai-trust-basic-act-20676-para-3', 'document_id': 'ai-trust-basic-act-20676', 'section': None, 'text': '제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다.<개정 2026. 1. 20.> [시행일: 2026. 1. 24.] 제2조제4호라목 중 디지털의료기기에 관한 부분', 'length': 96, 'parent_id': None}]


### 7.6 법률 구조 파싱 [필수]

정규식으로 `제n장`, `제n조`, `제n조의m`, `부칙`을 인식한다. 본칙과 부칙은 별도
ID 접두사를 사용한다. 조문 안의 원문 문단 순서는 그대로 유지한다.


In [10]:
CHAPTER_RE = re.compile(r"^제(?P<number>\d+)장\s+(?P<title>.+)$")
ADDENDUM_RE = re.compile(r"^부칙(?:\s|<|$)")
ARTICLE_RE = re.compile(
    r"^제(?P<number>\d+)조(?:의(?P<sub_number>\d+))?"
    r"\((?P<title>[^)]+)\)\s*(?P<body>.*)$"
)
PARAGRAPH_RE = re.compile(r"^[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]")
ITEM_RE = re.compile(r"^\d+(?:의\d+)?\.\s*")
SUBITEM_RE = re.compile(r"^[가-하]\.\s*")


def parse_law_articles(paragraphs: list[str]) -> list[dict]:
    """장·조문·부칙 경계를 인식해 법률을 조문 레코드로 변환한다."""
    articles: list[dict] = []
    chapter = ""
    scope = "본칙"
    current: dict | None = None

    def flush_current() -> None:
        nonlocal current
        if current is None:
            return
        current["text"] = "\n".join(current["lines"])
        current["length"] = len(current["text"])
        articles.append(current)
        current = None

    for paragraph in paragraphs:
        chapter_match = CHAPTER_RE.match(paragraph)
        if chapter_match:
            flush_current()
            scope = "본칙"
            chapter = paragraph
            continue

        if ADDENDUM_RE.match(paragraph):
            flush_current()
            scope = "부칙"
            chapter = paragraph
            continue

        article_match = ARTICLE_RE.match(paragraph)
        if article_match:
            flush_current()
            number = article_match.group("number")
            sub_number = article_match.group("sub_number") or ""
            article_label = f"제{number}조" + (f"의{sub_number}" if sub_number else "")
            scope_key = "main" if scope == "본칙" else "addendum"
            article_key = number + (f"-{sub_number}" if sub_number else "")
            article_id = f"{LAW_ID}-{scope_key}-article-{article_key}"
            title = article_match.group("title").strip()
            current = {
                "article_id": article_id,
                "document_id": LAW_ID,
                "law_name": LAW_NAME,
                "scope": scope,
                "chapter": chapter,
                "article": article_label,
                "article_number": article_key,
                "article_title": title,
                "section": f"{article_label}({title})",
                "source_file": SOURCE_FILE,
                "effective_date": LAW_EFFECTIVE_DATE,
                "lines": [paragraph],
            }
        elif current is not None:
            current["lines"].append(paragraph)

    flush_current()
    return articles


ARTICLES = parse_law_articles(LAW_BODY_PARAGRAPHS)
MAIN_ARTICLES = [article for article in ARTICLES if article["scope"] == "본칙"]
ADDENDUM_ARTICLES = [article for article in ARTICLES if article["scope"] == "부칙"]

print(f"조문 수: {len(ARTICLES)}개 (본칙 {len(MAIN_ARTICLES)}개 / 부칙 {len(ADDENDUM_ARTICLES)}개)")
print("첫 조문:", MAIN_ARTICLES[0]["section"])
print("마지막 본칙 조문:", MAIN_ARTICLES[-1]["section"])


조문 수: 47개 (본칙 44개 / 부칙 3개)
첫 조문: 제1조(목적)
마지막 본칙 조문: 제43조(과태료)


### 7.7 법률 계층 기반 Retrieval Chunking [필수]

짧은 조문은 하나의 검색 Chunk로 유지한다. 법률명·장·조문 표제를 포함한 길이가
1,000자를 넘을 때만 항→호→목 경계로 내려간다. Child는 `parent_id`로 전체 조문을
참조하므로 답변 생성 시 Parent 조문 전체로 확장할 수 있다.


In [11]:
def _body_lines(article: dict) -> list[str]:
    """첫 줄의 조문 표제는 제거하고 조문 본문만 반환한다."""
    first_match = ARTICLE_RE.match(article["lines"][0])
    first_body = first_match.group("body").strip() if first_match else article["lines"][0]
    return ([first_body] if first_body else []) + article["lines"][1:]


def _split_groups(lines: list[str], marker: re.Pattern) -> tuple[list[str], list[list[str]]]:
    """표지 문맥과 marker로 시작하는 연속 그룹을 분리한다."""
    prelude: list[str] = []
    groups: list[list[str]] = []
    current: list[str] | None = None
    for line in lines:
        if marker.match(line):
            if current:
                groups.append(current)
            current = [line]
        elif current is None:
            prelude.append(line)
        else:
            current.append(line)
    if current:
        groups.append(current)
    return prelude, groups


def _semantic_units(article: dict, budget: int) -> list[tuple[str, list[str]]]:
    """긴 조문을 항→호→목 경계로만 내려가며 의미 단위로 분리한다."""
    body = _body_lines(article)
    if not body:
        return [("article", [])]

    if any(PARAGRAPH_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "paragraph", PARAGRAPH_RE, "item", ITEM_RE
        )
    elif any(ITEM_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "item", ITEM_RE, "subitem", SUBITEM_RE
        )
    else:
        return [("article_part", body)]

    shared_intro, primary_groups = _split_groups(body, primary_marker)
    units: list[tuple[str, list[str]]] = []
    for primary_group in primary_groups:
        candidate = shared_intro + primary_group
        if len("\n".join(candidate)) <= budget:
            units.append((primary_type, candidate))
            continue

        local_intro, secondary_groups = _split_groups(primary_group, secondary_marker)
        if not secondary_groups:
            for line in primary_group:
                units.append((primary_type, shared_intro + [line]))
            continue

        for secondary_group in secondary_groups:
            secondary_candidate = shared_intro + local_intro + secondary_group
            if len("\n".join(secondary_candidate)) <= budget:
                units.append((secondary_type, secondary_candidate))
                continue

            tertiary_intro, tertiary_groups = _split_groups(secondary_group, SUBITEM_RE)
            if tertiary_groups:
                for tertiary_group in tertiary_groups:
                    units.append(("subitem", shared_intro + local_intro + tertiary_intro + tertiary_group))
            else:
                for line in secondary_group:
                    units.append((secondary_type, shared_intro + local_intro + [line]))
    return units or [("article_part", body)]


def _base_metadata(article: dict) -> dict:
    return {
        "document_id": article["document_id"],
        "law_name": article["law_name"],
        "scope": article["scope"],
        "chapter": article["chapter"],
        "article": article["article"],
        "article_number": article["article_number"],
        "article_title": article["article_title"],
        "section": article["section"],
        "source_file": article["source_file"],
        "effective_date": article["effective_date"],
        "hierarchy_path": f"{article['scope']} > {article['chapter']} > {article['section']}",
    }


def _context_prefix(article: dict) -> str:
    return f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['section']}"


def build_parent_documents(articles: list[dict]) -> list[dict]:
    """답변 시 전체 조문 맥락으로 확장할 Parent 문서를 만든다."""
    parents = []
    for article in articles:
        text = f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['text']}"
        parents.append({
            "doc_id": article["article_id"],
            **_base_metadata(article),
            "chunk_type": "article_parent",
            "parent_id": "",
            "text": text,
            "length": len(text),
        })
    return parents


def build_retrieval_documents(articles: list[dict], max_chars: int = 1000) -> list[dict]:
    """짧은 조문은 그대로, 긴 조문은 법률 계층 경계로 분할한다."""
    retrieval_documents: list[dict] = []
    for article in articles:
        prefix = _context_prefix(article)
        body_text = "\n".join(_body_lines(article))
        contextual_full_text = prefix + (f"\n{body_text}" if body_text else "")
        if len(contextual_full_text) <= max_chars:
            retrieval_documents.append({
                "doc_id": article["article_id"],
                **_base_metadata(article),
                "chunk_type": "article",
                "parent_id": "",
                "text": contextual_full_text,
                "length": len(contextual_full_text),
            })
            continue

        budget = max(200, max_chars - len(prefix) - 1)
        units = _semantic_units(article, budget)
        for index, (chunk_type, unit_lines) in enumerate(units, start=1):
            text = prefix + ("\n" + "\n".join(unit_lines) if unit_lines else "")
            retrieval_documents.append({
                "doc_id": f"{article['article_id']}-part-{index:02d}",
                **_base_metadata(article),
                "chunk_type": chunk_type,
                "parent_id": article["article_id"],
                "text": text,
                "length": len(text),
            })
    return retrieval_documents


PARENT_DOCUMENTS = build_parent_documents(ARTICLES)
RETRIEVAL_DOCUMENTS = build_retrieval_documents(ARTICLES, max_chars=1000)
SAMPLE_DOCUMENTS = RETRIEVAL_DOCUMENTS
PARENT_BY_ID = {document["doc_id"]: document for document in PARENT_DOCUMENTS}

long_article_children = [document for document in RETRIEVAL_DOCUMENTS if document["parent_id"]]
print(f"Parent 조문: {len(PARENT_DOCUMENTS)}개")
print(f"검색 Chunk: {len(RETRIEVAL_DOCUMENTS)}개 / 장문 조문 Child: {len(long_article_children)}개")
print("Chunk 유형:", sorted({document["chunk_type"] for document in RETRIEVAL_DOCUMENTS}))
structure_chunks = PARENT_DOCUMENTS
rag_chunks = RETRIEVAL_DOCUMENTS
print(rag_chunks[0])


Parent 조문: 47개
검색 Chunk: 96개 / 장문 조문 Child: 56개
Chunk 유형: ['article', 'item', 'paragraph']
{'doc_id': 'ai-trust-basic-act-20676-main-article-1', 'document_id': 'ai-trust-basic-act-20676', 'law_name': '인공지능 발전과 신뢰 기반 조성 등에 관한 기본법', 'scope': '본칙', 'chapter': '제1장 총칙', 'article': '제1조', 'article_number': '1', 'article_title': '목적', 'section': '제1조(목적)', 'source_file': '인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx', 'effective_date': '2026. 1. 22.', 'hierarchy_path': '본칙 > 제1장 총칙 > 제1조(목적)', 'chunk_type': 'article', 'parent_id': '', 'text': '인공지능 발전과 신뢰 기반 조성 등에 관한 기본법\n본칙 | 제1장 총칙\n제1조(목적)\n이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.', 'length': 156}


### 7.8 Parent-Child 조문 구조 [필수]

`PARENT_DOCUMENTS`는 조문 전체를, `RETRIEVAL_DOCUMENTS`는 실제 검색 단위를
나타낸다. 긴 조문의 Child가 검색되면 `PARENT_BY_ID[parent_id]`로 전체 조문을
복원할 수 있다.


In [12]:
child_chunks = [chunk for chunk in RETRIEVAL_DOCUMENTS if chunk["parent_id"]]
parent_child_chunks = PARENT_DOCUMENTS + child_chunks

print(f"Parent 조문: {len(PARENT_DOCUMENTS)}개")
print(f"장문 조문 Child: {len(child_chunks)}개")
if child_chunks:
    sample_child = child_chunks[0]
    sample_parent = PARENT_BY_ID[sample_child["parent_id"]]
    print("Child:", sample_child["hierarchy_path"], sample_child["chunk_type"])
    print("복원 Parent:", sample_parent["section"])


Parent 조문: 47개
장문 조문 Child: 56개
Child: 본칙 > 제1장 총칙 > 제2조(정의) item
복원 Parent: 제2조(정의)


## 8. 실행 결과 관찰

글자 수 기반 비교 전략과 법률 구조 기반 전략의 Chunk 수·길이를 비교한다.
구조 기반 결과는 법적 인용에 필요한 metadata가 모두 채워져 있다는 점이 핵심이다.


In [13]:
strategy_results = {
    "고정 길이(1000자)": chunk_by_fixed_length(document_text, LAW_ID, chunk_size=1000),
    "Overlap(1000자, 150자)": chunk_by_fixed_length_with_overlap(document_text, LAW_ID, chunk_size=1000, overlap=150),
    "조문 Parent": PARENT_DOCUMENTS,
    "법률 계층 Retrieval": RETRIEVAL_DOCUMENTS,
}

summary_rows = []
for name, chunks in strategy_results.items():
    lengths = [chunk["length"] for chunk in chunks]
    summary_rows.append({
        "전략": name,
        "chunk_수": len(chunks),
        "평균_길이": round(sum(lengths) / len(lengths), 1),
        "최대_길이": max(lengths),
        "최소_길이": min(lengths),
        "조문_metadata": all("article" in chunk for chunk in chunks),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df


,전략,chunk_수,평균_길이,최대_길이,최소_길이,조문_metadata
0,고정 길이(1000자),25,991.9,1000,798,False
1,"Overlap(1000자, 150자)",29,999.9,1000,998,False
2,조문 Parent,47,576.9,2047,137,True
3,법률 계층 Retrieval,96,330.9,932,112,True


**결과 해석**: 고정 길이와 Overlap 전략은 조문 번호·장·본칙/부칙 정보를 알지 못한다.
법률 계층 Retrieval은 짧은 조문을 보존하고 긴 조문만 세분화하며, 모든 결과에
`article`, `chapter`, `scope`, `hierarchy_path`가 존재한다.


## 9. 핵심 비교 실험

### 9.1 Chunk Size 변경 [필수]


In [14]:
for size in [400, 1000, 2000]:
    chunks = chunk_by_fixed_length(document_text, LAW_ID, chunk_size=size)
    split_article_headers = sum(
        1 for chunk in chunks
        if chunk["text"].startswith(("조", "의", "항", "에", "를", "및"))
    )
    print(f"chunk_size={size:4d} → {len(chunks):3d}개 | 문맥 없는 시작 후보={split_article_headers}")


chunk_size= 400 →  62개 | 문맥 없는 시작 후보=3
chunk_size=1000 →  25개 | 문맥 없는 시작 후보=2
chunk_size=2000 →  13개 | 문맥 없는 시작 후보=1


**관찰**: 크기만 키우면 조문이 덜 잘리지만 여러 조문이 한 Chunk에 섞인다. 크기를
줄이면 항·호 중간에서 시작하는 Chunk가 늘어난다. 법률 경계를 모르는 한 적절한
숫자 하나로 두 문제를 동시에 해결할 수 없다.


### 9.2 Overlap 변경 [필수]


In [15]:
no_overlap = chunk_by_fixed_length_with_overlap(document_text, LAW_ID, chunk_size=1000, overlap=0)
with_overlap = chunk_by_fixed_length_with_overlap(document_text, LAW_ID, chunk_size=1000, overlap=150)
print(f"overlap=0   → {len(no_overlap)}개")
print(f"overlap=150 → {len(with_overlap)}개")
print("중복 글자 증가:", sum(c["length"] for c in with_overlap) - len(document_text))


overlap=0   → 25개
overlap=150 → 29개
중복 글자 증가: 4200


**관찰**: Overlap은 경계 손실을 완화하지만 어느 조문에 속하는지 알려주지 못하며,
동일 법문을 여러 번 검색할 가능성을 높인다. 법률 metadata를 대체할 수 없다.


### 9.3 제목과 본문 분리 문제 [필수]


In [16]:
article34 = next(article for article in ARTICLES if article["scope"] == "본칙" and article["article"] == "제34조")
broken_demo = chunk_by_fixed_length(article34["text"], "demo", chunk_size=120)
print("고정 길이 첫 두 Chunk:")
for chunk in broken_demo[:2]:
    print(repr(chunk["text"]))

legal_demo = [chunk for chunk in RETRIEVAL_DOCUMENTS if chunk["article"] == "제34조"]
print("법률 구조 기반:")
for chunk in legal_demo:
    print(chunk["hierarchy_path"], "|", chunk["chunk_type"], "|", chunk["length"])


고정 길이 첫 두 Chunk:
'제34조(고영향 인공지능과 관련한 사업자의 책무) ① 인공지능사업자는 고영향 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 고영향 인공지능의 안전성ㆍ신뢰성을 확보하기 위하여 다음 각 호의 내용을 포함하는 조'
'치를 대통령령으로 정하는 바에 따라 이행하여야 한다.\n1. 위험관리방안의 수립ㆍ운영\n2. 기술적으로 가능한 범위에서의 인공지능이 도출한 최종결과, 인공지능의 최종결과 도출에 활용된 주요 기준, 인공지능의 개발ㆍ활용에'
법률 구조 기반:
본칙 > 제4장 인공지능윤리 및 신뢰성 확보 > 제34조(고영향 인공지능과 관련한 사업자의 책무) | article | 636


**관찰**: 고정 길이는 `제34조` 표제와 개별 의무를 분리할 수 있다. 구조 기반 Chunk는
검색 단위마다 제4장·제34조·조문 제목을 반복하고, 장문일 때만 항·호를 분리한다.


### 9.4 문맥 단절 문제 [선택: 추가 관찰]


In [17]:
article2 = next(article for article in ARTICLES if article["scope"] == "본칙" and article["article"] == "제2조")
article2_chunks = [chunk for chunk in RETRIEVAL_DOCUMENTS if chunk["article"] == "제2조"]
print("제2조 전체 길이:", article2["length"])
for chunk in article2_chunks[:5]:
    print(chunk["chunk_type"], "|", chunk["section"], "|", chunk["text"][-90:])


제2조 전체 길이: 2007
item | 제2조(정의) | 목 중 디지털의료기기에 관한 부분
1. “인공지능”이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
item | 제2조(정의) | 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
item | 제2조(정의) |  제2조제4호라목 중 디지털의료기기에 관한 부분
3. “인공지능기술”이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
item | 제2조(정의) | 른 유아교육ㆍ초등교육 및 중등교육에서의 학생 평가
카.그 밖에 사람의 생명ㆍ신체의 안전 및 기본권 보호에 중대한 영향을 미치는 영역으로서 대통령령으로 정하는 영역
item | 제2조(정의) |  제2조제1호에 따른 데이터를 말한다. 이하 같다)의 구조와 특성을 모방하여 글, 소리, 그림, 영상, 그 밖의 다양한 결과물을 생성하는 인공지능시스템을 말한다.


**관찰**: 정의 조문은 여러 용어를 호 단위로 열거하므로 전체 조문 하나보다 호 단위
검색이 정밀하다. 다만 모든 Chunk에 `제2조(정의)`가 반복되어 정의의 법적 출처가
유지된다.


### 9.5 지나치게 큰 Chunk 문제 [선택: 추가 관찰]


In [18]:
fixed_huge = chunk_by_fixed_length(document_text, LAW_ID, chunk_size=4000)
for chunk in fixed_huge[:2]:
    article_count = len(ARTICLE_RE.findall(chunk["text"]))
    print(f"고정 길이 Chunk 안 조문 표제 수: {article_count}")

mixed_legal_chunks = [
    chunk for chunk in RETRIEVAL_DOCUMENTS
    if len(set(match.group(0) for match in ARTICLE_RE.finditer(chunk["text"]))) > 1
]
print("법률 구조 Chunk 중 여러 조문이 섞인 수:", len(mixed_legal_chunks))


고정 길이 Chunk 안 조문 표제 수: 0
고정 길이 Chunk 안 조문 표제 수: 0
법률 구조 Chunk 중 여러 조문이 섞인 수: 0


**관찰**: 큰 고정 Chunk에는 여러 조문이 섞여 검색 결과의 정확한 인용 범위가
불분명해진다. 법률 구조 Chunk는 한 Chunk가 정확히 한 조문에만 속한다.


## 10. 구조 복원 검증

Child 검색 결과에서 `parent_id`로 전체 조문을 복원하고, 본칙/부칙의 같은 조 번호가
서로 다른 ID를 갖는지 확인한다.


In [19]:
sample_child = next(chunk for chunk in RETRIEVAL_DOCUMENTS if chunk["parent_id"])
restored_parent = PARENT_BY_ID[sample_child["parent_id"]]
print("검색 Child:", sample_child["doc_id"])
print("복원 Parent:", restored_parent["doc_id"], restored_parent["section"])

main_article1 = next(doc for doc in PARENT_DOCUMENTS if doc["scope"] == "본칙" and doc["article"] == "제1조")
addendum_article1 = next(doc for doc in PARENT_DOCUMENTS if doc["scope"] == "부칙" and doc["article"] == "제1조")
print("본칙 제1조 ID:", main_article1["doc_id"])
print("부칙 제1조 ID:", addendum_article1["doc_id"])
assert main_article1["doc_id"] != addendum_article1["doc_id"]


검색 Child: ai-trust-basic-act-20676-main-article-2-part-01
복원 Parent: ai-trust-basic-act-20676-main-article-2 제2조(정의)
본칙 제1조 ID: ai-trust-basic-act-20676-main-article-1
부칙 제1조 ID: ai-trust-basic-act-20676-addendum-article-1


**결과 해석**: 구조 기반 Chunking은 섹션 경계를 알고 자르기 때문에 하나의 Chunk에
여러 주제가 섞이지 않고, 제목과 본문이 항상 함께 유지된다. 다만 이 방식은 문서에
`"숫자. 제목"` 같은 뚜렷한 구조가 있어야만 동작한다.


## 11. 도전 과제

1. 인용된 타 법률명을 별도 metadata로 추출해 법령 간 그래프를 만든다.
2. 개정문과 시행일 문구를 조문 metadata로 분리한다.
3. 검색된 Child를 Parent 조문으로 확장한 경우와 그대로 답변한 경우를 비교한다.


## 12. 테스트

실제 HWPX 파싱 결과, 법률 계층, ID 유일성, 장문 조문 분할과 Parent 복원을 검증한다.


In [20]:
assert len(ARTICLES) == 47
assert len(MAIN_ARTICLES) == 44
assert len(ADDENDUM_ARTICLES) == 3
assert any(article["article"] == "제22조의2" for article in MAIN_ARTICLES)
assert len({article["article_id"] for article in ARTICLES}) == len(ARTICLES)

assert len(PARENT_DOCUMENTS) == len(ARTICLES)
assert RETRIEVAL_DOCUMENTS
assert len({chunk["doc_id"] for chunk in RETRIEVAL_DOCUMENTS}) == len(RETRIEVAL_DOCUMENTS)
assert all(chunk["document_id"] == LAW_ID for chunk in RETRIEVAL_DOCUMENTS)
assert all(chunk["chapter"] and chunk["article"] for chunk in RETRIEVAL_DOCUMENTS)
assert all(chunk["scope"] in {"본칙", "부칙"} for chunk in RETRIEVAL_DOCUMENTS)
assert all(chunk["length"] <= 1000 for chunk in RETRIEVAL_DOCUMENTS)

article2_chunks = [chunk for chunk in RETRIEVAL_DOCUMENTS if chunk["article"] == "제2조" and chunk["scope"] == "본칙"]
assert len(article2_chunks) > 1
assert all(chunk["parent_id"] for chunk in article2_chunks)
assert any("고영향 인공지능" in chunk["text"] for chunk in article2_chunks)

main_ids = {doc["doc_id"] for doc in PARENT_DOCUMENTS if doc["scope"] == "본칙"}
addendum_ids = {doc["doc_id"] for doc in PARENT_DOCUMENTS if doc["scope"] == "부칙"}
assert main_ids.isdisjoint(addendum_ids)
assert all(chunk["parent_id"] in PARENT_BY_ID for chunk in RETRIEVAL_DOCUMENTS if chunk["parent_id"])
print("HWPX 법률 구조 Chunking 테스트 통과")


HWPX 법률 구조 Chunking 테스트 통과


## 13. 결과 저장


In [21]:
chunking_log = {
    "source_file": SOURCE_FILE,
    "law_name": LAW_NAME,
    "effective_date": LAW_EFFECTIVE_DATE,
    "article_count": len(ARTICLES),
    "main_article_count": len(MAIN_ARTICLES),
    "addendum_article_count": len(ADDENDUM_ARTICLES),
    "retrieval_chunk_count": len(RETRIEVAL_DOCUMENTS),
    "child_chunk_count": len([c for c in RETRIEVAL_DOCUMENTS if c["parent_id"]]),
    "chunk_types": sorted({c["chunk_type"] for c in RETRIEVAL_DOCUMENTS}),
    "sample_chunks": RETRIEVAL_DOCUMENTS[:5],
}
saved_path = save_log(
    chunking_log,
    OUTPUT_DIR / "logs" / "aitrust_1_document_chunking_log.json",
)
print("저장 위치:", saved_path)


저장 위치: E:\agentic_ai_lab\outputs\logs\aitrust_1_document_chunking_log.json


## 14. 핵심 정리

- HWPX는 ZIP 내부 `Contents/section*.xml`의 문단을 순서대로 읽는다.
- 법률 검색의 기본 단위는 조문이며 본칙과 부칙의 ID 공간을 분리한다.
- 짧은 조문은 보존하고 긴 조문만 항→호→목 경계에서 세분화한다.
- Child에 법률명·장·조문 표제와 `parent_id`를 보존해 정확한 인용과 문맥 확장이 가능하다.
- Overlap은 법률 계층 metadata를 대체하지 못한다.


## 15. 확인 문제

1. 본칙 제1조와 부칙 제1조의 ID를 분리해야 하는 이유는 무엇인가?
2. 긴 조문을 문장 단위가 아니라 항→호→목 순서로 나누는 이유는 무엇인가?
3. 검색 Child에 조문 표제를 반복하는 이유는 무엇인가?
4. `parent_id`는 답변 생성 단계에서 어떻게 사용할 수 있는가?
